## 14. 子 Agent：把专项任务外包

> 来源：[Subagents in the SDK](https://code.claude.com/docs/en/agent-sdk/subagents)、[Create custom subagents](https://code.claude.com/docs/en/sub-agents)

主 agent 可以派生**专职子 agent** 做聚焦子任务。
三大价值：
- **context 隔离**（中间 tool call/result 留在子 agent 里，只有最终消息回到父级——父级 context 只增长一份摘要）；
- **并行**（多个子 agent 并发，总耗时约等于最慢者）；
- **专用指令 + 收窄工具集**；

定义子 agent 有三条路：`agents` 参数**编程式定义**（SDK 应用推荐）、`.claude/agents/` 目录下的 **markdown 文件式定义**、以及什么都不定义、直接用**内置的 `general-purpose`**（适合临时的研究/探索委派）。同名冲突时编程式定义覆盖文件式定义。

子 agent 通过内建的 `Agent` tool 调用，所以要把 `"Agent"` 加进 `allowed_tools` 才能免审批派发——不加的话，派发请求会落到 `can_use_tool` 回调上，`dontAsk` 权限模式下则直接被拒，表现为"Claude 不委派、自己干了"。Claude 依据每个子 agent 的 `description` 自主决定是否委派；prompt 里点名（"Use the code-reviewer agent to..."）可强制调用。下方 cell 是一次最小派发（定义 `poem-style` 子 agent 写诗，打印原始消息流），本章各节引用的真实消息都出自这类运行。

派发默认**后台运行**（🆕）。同步执行是指 `Agent` tool 调用一直阻塞，等子 agent 跑完，把最终消息作为 tool result 返回；后台运行是指调用立即返回，主 agent 接着干别的，子 agent 完成后结果再送回来。省略 `run_in_background` 参数时就按后台派发，Claude 需要先拿到结果才能继续时，会显式传 `run_in_background: false`；`AgentDefinition` 的 `background=True` 则强制该 agent 始终后台运行，不管 Claude 传了什么。

子 agent 可以再派子 agent，最深嵌套 5 层，前台后台都计入层数；不想让它继续往下派，就把 `Agent` 排除在它的 `tools` 之外，或加进 `disallowedTools`。超大规模编排（几十到几百个 agent）用 `Workflow` tool，目前仅 TypeScript SDK 提供，同样要加进 `allowedTools` 才免审批。

In [1]:
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition


async def main():
    async for message in query(
        prompt="Please write a poem using 'poem-style'",
        options=ClaudeAgentOptions(
            allowed_tools=["Agent"],
            agents={
                "poem-style": AgentDefinition(
                    description="Poem style guild",
                    prompt="Poem should contains '梁柱', and in Chinese",
                    tools=["Read", "Glob", "Grep"],
                )
            },
        ),
    ):
        print(message)


await main()

HookEventMessage(subtype='hook_started', data={'type': 'system', 'subtype': 'hook_started', 'hook_id': '10568d33-b412-4b14-b5a2-849ebd190544', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'uuid': 'fac09be9-630b-4cb8-8c0c-30bcd2d378a4', 'session_id': '1112077a-372f-4d0a-b155-4cc0232c7ade'}, hook_event_name='SessionStart', session_id='1112077a-372f-4d0a-b155-4cc0232c7ade', uuid='fac09be9-630b-4cb8-8c0c-30bcd2d378a4')
HookEventMessage(subtype='hook_response', data={'type': 'system', 'subtype': 'hook_response', 'hook_id': '10568d33-b412-4b14-b5a2-849ebd190544', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'output': '', 'stdout': '', 'stderr': '', 'exit_code': 0, 'outcome': 'success', 'uuid': 'a38e717f-d7b4-4d44-9784-e48593408570', 'session_id': '1112077a-372f-4d0a-b155-4cc0232c7ade'}, hook_event_name='SessionStart', session_id='1112077a-372f-4d0a-b155-4cc0232c7ade', uuid='a38e717f-d7b4-4d44-9784-e48593408570')
SystemMessage(subtype='init', data={

### 14.1 编程式定义（推荐）

定义子 agent 就是往 `agents` dict 里放 `AgentDefinition`。下方 cell 是官方的标准示例：一次定义两个专职 agent——只读的代码审查者、能跑命令的测试执行者，Claude 按各自的 `description` 匹配任务决定派给谁。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition


async def demo_define_subagents():
    async for message in query(
        prompt="Review the authentication module for security issues",
        options=ClaudeAgentOptions(
            # 自动批准名单必须含 "Agent"，否则派发要走审批流程
            allowed_tools=["Read", "Grep", "Glob", "Agent"],
            agents={
                "code-reviewer": AgentDefinition(
                    # description 决定 Claude 何时委派给它——写清楚适用场景
                    description="Expert code review specialist. Use for quality, "
                                "security, and maintainability reviews.",
                    # prompt 是它的 system prompt：人设与工作方式
                    prompt="You are a code review specialist with expertise in "
                           "security, performance, and best practices.\n\n"
                           "When reviewing code:\n"
                           "- Identify security vulnerabilities\n"
                           "- Check for performance issues\n"
                           "- Suggest specific improvements\n\n"
                           "Be thorough but concise in your feedback.",
                    tools=["Read", "Grep", "Glob"],   # 只读收窄：能看不能改
                    model="sonnet",                   # 子任务用小模型省成本
                ),
                "test-runner": AgentDefinition(
                    description="Runs and analyzes test suites. Use for test "
                                "execution and coverage analysis.",
                    prompt="You are a test execution specialist. Run tests and "
                           "provide clear analysis of results: identify failing "
                           "tests and suggest fixes.",
                    tools=["Bash", "Read", "Grep"],   # 有 Bash 才能跑测试命令
                    disallowedTools=["Agent"],        # camelCase 字段；禁止它再往下派子 agent
                ),
            },
        ),
    ):
        if hasattr(message, "result"):
            print(message.result)


await demo_define_subagents()

`AgentDefinition` 全部字段：

| 字段 | 类型 | 说明 |
|---|---|---|
| `description` | `str`（必填） | 何时用这个 agent——Claude 靠它决定委派 |
| `prompt` | `str`（必填） | 该 agent 的专属 system prompt |
| `tools` | `list[str]` | 允许的工具；省略则继承父级全部 |
| `disallowedTools` | `list[str]` | 移除工具；支持 `mcp__server__*` / `mcp__*` 模式 |
| `model` | `str` | 模型覆盖：`'fable'`/`'opus'`/`'sonnet'`/`'haiku'`/`'inherit'` 或完整 model ID |
| `effort` | `str` | 按 agent 覆盖推理深度 |
| `permissionMode` | `str` | 该 agent 内的权限模式（父级 `bypassPermissions`/`acceptEdits` 时会被继承且不可覆盖） |
| `maxTurns` / `background` / `skills` / `memory` / `mcpServers` / `initialPrompt` | — | 轮次上限 / 后台运行 / 预载技能 / memory 来源 / 专属 MCP / 主线程首条输入 |

> [!warning] Python 命名坑
> `AgentDefinition` 的多词字段**保持 camelCase**（`disallowedTools`、`mcpServers`、`maxTurns`、`permissionMode`），不遵循 snake_case——它们直接映射 wire 格式（SDK 与 CLI 子进程之间传输的原始 JSON）的键名。上方 cell 里 `disallowedTools=["Agent"]` 就是实际写法。

`tools` 的常用组合（对应上方两个 agent 的取舍）：

| 用途 | tools | 效果 |
|---|---|---|
| 只读分析 | `Read`, `Grep`, `Glob` | 能审查，不能改文件、不能执行命令 |
| 测试执行 | `Bash`, `Read`, `Grep` | 能跑命令、分析输出 |
| 改代码 | `Read`, `Edit`, `Write`, `Grep`, `Glob` | 全读写，但不能执行命令 |
| 全量 | 省略 `tools` 字段 | 继承父级全部工具 |

文件式定义（`.claude/agents/*.md`）的一个加载坑：目录监听只覆盖 session 启动时已存在的目录——**第一次新建 `agents` 目录后要重启 session** 新定义才会被发现；之后新增/修改文件几秒内自动生效。

### 14.1.1 按运行时条件动态生成定义

`AgentDefinition` 在 `query()` 调用时才求值，所以可以用工厂函数按运行时条件（用户等级、任务风险、成本预算）现场生成——同一个 agent 名字，每次请求可以是不同的 prompt 和模型：

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition


def create_security_agent(security_level: str) -> AgentDefinition:
    """工厂函数：按运行时条件生成 AgentDefinition。"""
    is_strict = security_level == "strict"
    return AgentDefinition(
        description="Security code reviewer",
        prompt=f"You are a {'strict' if is_strict else 'balanced'} security "
               "reviewer. Flag every injection risk, auth flaw, and unsafe "
               "dependency you find.",
        tools=["Read", "Grep", "Glob"],
        model="opus" if is_strict else "sonnet",  # 高风险审查换更强的模型
    )


async def demo_dynamic_agent():
    async for message in query(
        prompt="Review this PR for security issues",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Grep", "Glob", "Agent"],
            # 定义在 query 时才生成，每次请求可用不同配置
            agents={"security-reviewer": create_security_agent("strict")},
        ),
    ):
        if hasattr(message, "result"):
            print(message.result)


await demo_dynamic_agent()

### 14.2 子 agent 的信息边界

子 agent 是一个 context 为空的全新对话，它没有独立的"接收用户输入"渠道。它的初始 context 里只有两段文字，来源和角色不同：

- `AgentDefinition.prompt`——开发者定义 agent 时写死的，充当子 agent 的 **system prompt**，写长期人设与工作方式（"你是谁、怎么干活"），每次派发都相同；
- `Agent` tool 调用的 `prompt` 参数——主 agent 每次派发时现写的任务指令，成为子 agent 的**首条用户消息**（"这次具体干什么"），每次派发都不同。

"现写"是指主 agent 不把用户原话原样转发，而是针对本次任务当场撰写指令，通常会扩写要求、追加输出约束。对照一次真实派发的消息流：用户对主 agent 只说了"写一首关于猫的诗"，主 agent 发出的 `Agent` tool 调用里，`prompt` 参数已经是扩写后的指令：

```python
ToolUseBlock(
    name="Agent",
    input={
        "subagent_type": "poem-style",
        "prompt": "Write an original poem about a cat. Make it evocative and "
                  "well-crafted, with vivid imagery ... Return only the finished poem.",
    },
)
```

紧接着，子 agent 的对话以一条 `UserMessage` 开场——内容与上面的 `prompt` 参数一字不差，`parent_tool_use_id` 指回这次 `Agent` tool 调用。在子 agent 看来这条消息就是用户输入，尽管实际撰写者是主 agent：

```python
UserMessage(
    content=[TextBlock(text="Write an original poem about a cat. Make it evocative ...")],
    parent_tool_use_id="toolu_01Lud...",  # 派生该子 agent 的那次 Agent tool 调用的 id
)
```

这个 `prompt` 参数是父级 → 子 agent 的**唯一通道**：父级 context 里的文件路径、报错、已做的决策，只有被主 agent 写进这个参数，子 agent 才看得见。Windows 下这段指令过长会撞上命令行 8191 字符的上限、导致派发失败——太长的指令改写进文件式定义。

| 子 agent 拿得到 | 拿不到 |
|---|---|
| system prompt（`AgentDefinition.prompt`）+ 派发任务指令（`Agent` tool 的 `prompt` 参数） | 父级对话历史与 tool result |
| 项目 CLAUDE.md（经 `setting_sources`） | 父级的 system prompt |
| 继承或收窄后的工具集 | 未列进 `skills` 的技能全文 |
| 主会话的 extended thinking 配置（🆕） | 父级会话里的临时权限批准 |

用户在父级会话里临时批准过的工具，子 agent 使用时会重新弹权限提示——派出的子 agent 越多，同样的提示重复越多次。要避免，就用 `PreToolUse` hook 或权限规则统一放行。

回传方向：子 agent 的最终消息**原样**作为 `Agent` tool result 交给父级，但父级在自己的回复里可能把它改写成摘要——要在最终输出里逐字保留子 agent 的结果，得在主 `query()` 的 prompt 或 `system_prompt` 里明确要求。

子 agent 因 API error（rate limit、服务端反复报错等）**提前终止**时，失败会如实上报，不会让错误文本冒充成子 agent 的结论（🆕）。具体形式取决于派发方式：

- **同步派发**：已经产出部分输出的，Agent tool result 返回这部分输出，并在后面附一句说明文字——子 agent 被中断、任务没有做完，提示父级这份输出是残缺的；什么都没产出的，这次 tool 调用直接报错，错误内容是 `Agent terminated early due to an API error` 加上具体错误。
- **后台派发**：子 agent 标记为失败，Claude 收到的结束通知里写明是哪种 API error，并附上子 agent 的最后输出，已完成的部分不会丢。

等 API error 消退后，可以让 Claude 重试任务，或 resume 该子 agent 接着跑。


### 14.2.1 识别消息归属：`parent_tool_use_id`

应用代码遍历 `query()` 的消息流时，主对话消息和子 agent loop 内部消息混在同一条流里。区分靠 `parent_tool_use_id`——SDK 附加在消息流对象上的归属元数据，不出现在给 LLM 的输入或 LLM 的响应里。规则：源自子 agent loop 的消息（子 agent 的 `AssistantMessage` 模型输出、子 agent loop 内作为 tool result 的 `UserMessage`）带此字段，值为派生该子 agent 的那次 `Agent` tool 调用的 `ToolUseBlock.id`；主对话消息该字段为 `None`。

仍以 poem 派发为例，一次派发在消息流里依次出现三类消息：

```python
# ① 主对话：主 agent 发出派发调用。这是主对话自己的消息，parent_tool_use_id 为 None
AssistantMessage(
    content=[ToolUseBlock(id="toolu_01Lud...", name="Agent", input={...})],
    parent_tool_use_id=None,
)

# ② 子 agent loop 内的消息：parent_tool_use_id 指回 ① 的 ToolUseBlock.id
UserMessage(
    content=[TextBlock(text="Write an original poem about a cat. ...")],
    parent_tool_use_id="toolu_01Lud...",
)

# ③ 主对话：子 agent 结果作为 tool result 回来。这又是主对话消息，parent_tool_use_id
#    回到 None；它和 ① 的对应关系记录在 ToolResultBlock.tool_use_id 里
UserMessage(
    content=[ToolResultBlock(tool_use_id="toolu_01Lud...", content=[...])],
    parent_tool_use_id=None,
)
```

检测"是否发生了派发"要看 `ToolUseBlock.name`，坑在于同一次会话里这个 tool 有两个名字：`tool_use` block 里发的是 `"Agent"`（🆕），但 `system:init` 的 tools 列表和 `result.permission_denials[].tool_name` 里仍是旧名 `"Task"`，所以**两个名字都要匹配**。同一次 poem 会话里两处并存：

```python
SystemMessage(subtype="init", data={"tools": ["Task", "Bash", ...]})      # 能力清单里叫 "Task"
ToolUseBlock(name="Agent", input={"subagent_type": "poem-style", ...})    # 实际派发时叫 "Agent"
```

下方 cell 是这套判断的可运行实现——输出里能看到派发时刻和一串来自子 agent 内部的消息：

In [1]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    ToolUseBlock,
    AssistantMessage,
    ResultMessage,
)


async def demo_subagents():
    async for message in query(
        prompt="Use the code-reviewer agent to review this codebase",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob", "Grep", "Agent"],  # 注意含 Agent
            agents={
                "code-reviewer": AgentDefinition(
                    description="Expert code reviewer for quality and security reviews.",
                    prompt="Analyze code quality and suggest improvements.",
                    tools=["Read", "Glob", "Grep"],  # 只读收窄
                    model="sonnet",  # 子任务用小模型省成本
                )
            },
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # tool_use block 里发 "Agent"，但部分字段仍用 "Task"，两个都匹配
                if isinstance(block, ToolUseBlock) and block.name in ("Agent", "Task"):
                    print("subagent invoked:", block.input.get("subagent_type"))
        # 源自子 agent loop 的消息对象带 parent_tool_use_id（主对话消息为 None）
        if getattr(message, "parent_tool_use_id", None):
            print("  (from inside subagent)")
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_subagents()

subagent invoked: code-reviewer
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside subagent)
  (from inside sub

### 14.2.2 存储隔离与跨重启 resume

子 agent 可以被**恢复（resume）**：不从零开始，而是带着上一次的完整对话——所有 tool call、tool result、推理——接着干活。能这么做，靠的是它的存储模型：一次会话在磁盘上不止一份 transcript，主对话一份，每个子 agent 各自独立一份；进程退出后内存里的对话没了，这些文件还在。compaction 压缩的也只是主对话自己的 context，子 agent 的 transcript 不受影响。

恢复要用两个 id，都藏在**第一次运行的消息流**里。以一次真实派发为例——用户要一首诗，主 agent 派 `poem-style` 子 agent 写完返回——流里这两处要存下来：

```python
# id 之一：agentId。子 agent 跑完后，Agent tool result 的 content 有两段文字：
# 第一段是子 agent 最终消息原文，第二段是 SDK 追加的续用说明，agentId 写在里面
ToolResultBlock(
    tool_use_id="toolu_vrtx_017X31...",
    content=[
        {"type": "text", "text": "## 风格约定\n\n**语言**\n- 中文书写，……"},
        {"type": "text", "text": "agentId: abbacb99b2fc86fc3 (use SendMessage "
                                 "with to: 'abbacb99b2fc86fc3', ...)"},   # ← agentId
    ],
)

# 后台派发时更好认：任务开始/结束通知里的 task_id 就是同一个 agentId
TaskStartedMessage(task_id="abbacb99b2fc86fc3", ...)

# id 之二：主会话的 session_id，在收尾的 ResultMessage 上
# （开场 system:init 的 SystemMessage 上也有同一个值）
ResultMessage(session_id="680f634f-d6a0-46b0-bb8e-c0f434b7aa4d", ...)
```

那段续用说明本身是写给主 agent 看的：**同一会话内**想继续用这个子 agent，主 agent 用 SendMessage 按 agentId 找它，不需要应用插手。**跨进程重启**才需要下面的 resume 流程——两个 id 只在消息流里出现，进程一退就没了，终端用户更不可能知道它们，所以第一次运行时就要由应用代码存进数据库或状态文件。

恢复分三步：

1. **第一次 `query()`**：遍历消息流，存下 `session_id` 和 agentId；
2. **第二次 `query()`**：options 传 `resume=session_id`——主会话从磁盘 transcript 加载，在原有对话基础上继续（不传 `resume` 的话每次 `query()` 都是全新会话）；
3. **prompt 里点名 agentId**：主 agent 看到点名，按该 id 恢复子 agent，子 agent 的 context 从它自己的 transcript 加载——它记得自己写过什么，不用重贴原文。终端用户只说业务话（"把那首猫诗改成英文版"），agentId 由应用从存储里取出、拼进 prompt。

完整代码见下方 cell。两次 `query()` 写在同一个函数里方便演示；拆成两个进程、隔一天再跑第二段也一样，只要两个 id 存了下来。

In [ ]:
import re
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    ToolResultBlock,
)

# 两次 query 必须传同一份 agents 定义：transcript 只存对话，
# 子 agent 的 system prompt 和工具集每次都从 AgentDefinition 现取
AGENTS = {
    "poem-style": AgentDefinition(
        description="Writes poems in a distinctive style.",
        prompt="You write restrained, imagery-driven free verse in Chinese.",
    )
}


def extract_agent_id(block: ToolResultBlock) -> str | None:
    """从 Agent tool result 的续用说明文字里解析 agentId。"""
    parts = (
        block.content if isinstance(block.content, list) else [{"text": block.content}]
    )
    for part in parts:
        if match := re.search(r"agentId:\s*([\w-]+)", part.get("text") or ""):
            return match.group(1)
    return None


async def demo_resume_subagent():
    agent_id = None
    session_id = None

    # 第一次运行：派发子 agent，顺手存两个 id
    async for message in query(
        prompt="用 poem-style agent 写一首关于猫的诗",
        options=ClaudeAgentOptions(allowed_tools=["Agent"], agents=AGENTS),
    ):
        if hasattr(message, "session_id"):
            session_id = message.session_id  # 存 id 之二
        for block in getattr(message, "content", None) or []:
            if isinstance(block, ToolResultBlock):
                agent_id = extract_agent_id(block) or agent_id  # 存 id 之一
        if hasattr(message, "result"):
            print(message.result)

    # ——进程在这里退出、第二天重启，都不影响下面这段，只要两个 id 存了下来——

    # 第二次运行：恢复主会话 + 点名恢复子 agent
    user_request = "把那首猫诗改成英文版"  # 终端用户的原话，只有业务意图
    async for message in query(
        prompt=f"Resume agent {agent_id}: {user_request}",  # 应用把 agentId 拼进 prompt
        options=ClaudeAgentOptions(
            allowed_tools=["Agent"],
            agents=AGENTS,  # 与第一次同一份定义
            resume=session_id,  # 恢复主会话
        ),
    ):
        if hasattr(message, "result"):
            print(message.result)


await demo_resume_subagent()

三条边界：

- **`agents` 定义要原样重传**——transcript 只存对话、不存 agent 定义，第二次 `query()` 不传或改了名，恢复失败。
- **内置 `Explore`/`Plan` 是 one-shot**——不返回 agentId、不能恢复；能恢复的是自定义 agent 和内置 `general-purpose`。
- **transcript 有保质期**——按 `cleanupPeriodDays` 清理（默认 30 天），清掉后无法恢复。


### 14.3 父子共享 state 的四条通道

context 隔离是特性不是缺陷——父子之间没有共享对话状态的机制，"共用 state"必须走对话之外的通道。按 state 的性质选：

| state 性质 | 通道 | 说明 |
|---|---|---|
| 静态共识（项目约定、术语） | CLAUDE.md | 双方都加载，每次请求重新注入，不受 compaction 影响 |
| 派发时的一次性输入（路径、报错、决策） | `Agent` tool 的 `prompt` 参数（即派发时的 `ToolUseBlock.input.prompt`，方向父 → 子） | 这些路径/报错在主 agent 的 context 里往往躺在它此前的 tool result 中，但子 agent 看不见父级对话历史——必须由主 agent 抄写进派发指令。软约束：主 agent 的 system prompt 里要求派发时带上；硬约束：`PreToolUse` hook 拦 `Agent` tool，用 `updatedInput` 把 state 强制拼进派发指令 |
| 动态可变 state（进度、共享结论、计数器） | **进程内 MCP tool（首选）** | handler 闭包引用同一个 Python 对象，父子都调同一组读写 tool；应用进程是单一事实源，可加锁、校验、审计 |
| 只需父侧聚合、子 agent 不用读 | `SubagentStart` / `SubagentStop` hook | 在应用进程收集各子 agent 结果，不占任何 agent 的 context |

进程内 MCP tool 当"共享内存"的骨架见下方 cell。

In [ ]:
from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    AgentDefinition,
)

shared_state = {"findings": []}  # 应用进程里的单一事实源


@tool("get_state", "Read shared findings", {})
async def get_state(args):
    return {"content": [{"type": "text", "text": str(shared_state["findings"])}]}


@tool("add_finding", "Append a finding to shared state", {"finding": str})
async def add_finding(args):
    shared_state["findings"].append(args["finding"])
    return {"content": [{"type": "text", "text": "ok"}]}


state_server = create_sdk_mcp_server(
    name="state", version="1.0.0", tools=[get_state, add_finding]
)

options = ClaudeAgentOptions(
    mcp_servers={"state": state_server},
    allowed_tools=["Agent", "mcp__state__*"],
    agents={
        "worker": AgentDefinition(
            description="Investigates one aspect and records findings to shared state.",
            prompt="Investigate the assigned aspect; record each finding via add_finding.",
            # 子 agent 的 tools 显式列上 state 工具（省略 tools 则继承全部，也能用）
            tools=["Read", "Grep", "mcp__state__get_state", "mcp__state__add_finding"],
        )
    },
)

备选是**文件系统黑板**：父子共享同一 `cwd`，state 写成 `state.json` 之类的文件，双方用 Read/Write 读写。零额外代码，但并行子 agent 写同一文件会竞态，且文件内容每次读取都消耗 context——留给"产物本身就是文件"的场景。

> [!warning] 两个边界
> fork 出的 session、并行的子 agent，隔离的都只是**对话**，文件系统始终共享——文件黑板天然可见，但也意味着写冲突要自己防。反过来，进程内 MCP tool 的 state 只活在应用进程里，进程重启即失；需要跨重启就把 tool 的存储换成 Redis/数据库。